---
Alumna: Sandra Geraldine Castillo Dorantes

## Sesion 11 - Ejercicio Integrador Final — Mini Pricing Actuarial

Construye el análisis actuarial completo de GMM en 4 fases.
Cada fase usa lo aprendido en esta sesión.

**El objetivo:** validar si la tarifa vigente de GMM está bien calibrada
respecto al riesgo real observado.

---

**Fase 1 — Parámetros técnicos por grupo de edad (NumPy)**

Para cada grupo de edad de GMM calcula:
`frecuencia`, `severidad`, `prima_pura` = frec × sev,
`prima_real` (promedio de prima_neta), `adecuacion` = prima_real / prima_pura.

¿Qué grupo está más sub-tarifado (adecuacion < 1)?

---

In [5]:
# LIBRERIAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Importamos archivo de cartera
ruta = "C:/Users/HP/diplomado-ml-seguros/Modulo_1/sesion_10"
cartera = pd.read_parquet(ruta + "/datos/cartera_q1_2026_final.parquet")

# Filtramos solamente el ramo de Gastos Medicos
gm = cartera[cartera['ramo'] == "GMM"]

# Creamos columna g_edad
gm['g_edad'] = pd.cut(gm['edad'], bins=[0,30,45,60,100],
                      labels=['18-30','31-45','46-60','61+'])

# Usamos Numpy para crear las columnas como arrays
primas     = np.array(gm['prima_total'])
siniestros = np.array(gm['n_siniestros'])
montos     = np.array(gm['monto_pagado'])
neta       = np.array(gm['prima_neta'])

grupos_edad = ['18-30', '31-45', '46-60', '61+']
resultados = []

# Filtramos con mascara
for grupo in grupos_edad:
    mask = np.array(gm['g_edad'] == grupo)
    p_grupo  = primas[mask]
    nt_grupo = neta[mask]
    s_grupo  = siniestros[mask]
    m_grupo  = montos[mask]
    
    n_pol = mask.sum()
    n_sin = s_grupo.sum()
    frec = n_sin / n_pol
    sev = m_grupo[m_grupo > 0].mean()
    p_pura = frec * sev
    p_real_n = nt_grupo.mean() # promedio de la prima neta
    p_real_t = p_grupo.mean() # promedio de la prima total
    adecuacion_n = p_real_n / p_pura # usando prima_neta
    adecuacion_t = p_real_t / p_pura # usando prima_total
    lr = m_grupo.sum() / p_grupo.sum()
    
    resultados.append({
        'grupo': grupo,
        'n_polizas': n_pol,
        'n_siniestros': n_sin,
        'frecuencia': frec,
        'severidad': sev,
        'prima_pura': p_pura,
        'prima_real_n': p_real_n, # usando prima_neta
        'prima_real_t': p_real_t, # usando prima_total
        'adecuacion': adecuacion_n, # usando prima_neta
        'adecuacion_t': adecuacion_t, # usando prima_total
        'loss_ratio': lr,
    })

tabla_F1 = pd.DataFrame(resultados)

print("=======================================")
print("PARAMETROS TECNICOS POR GRUPO DE EDAD")
print("=======================================")
print(tabla_F1)
print()

print("==========================")
print("        ADECUACION        ")
print("==========================")
sub = []
sobre = []
for i in resultados:
    if i['adecuacion'] > 1:
        sobre.append({'grupo':i['grupo'], 'adecuacion': i['adecuacion']})
    if i['adecuacion'] < 1:
        sub.append({'grupo':i['grupo'], 'adecuacion': i['adecuacion']})

if len(sobre) == 4:
    print("Ningun grupo esta subtarifado")
if len(sobre) > 1:
    print("Grupos sobretarifados")
    for i in sobre:
        print(f"Grupo: {i['grupo']:<6} | Adecuacion {i['adecuacion'].round(2)}")
if len(sub) == 4:
    print("Todos los grupos estan subtarifados")
if len(sub) > 1:
    print("Grupos subtarifados")
    for i in sub:
        print(f"Grupo: {i['grupo']:<6} | Adecuacion {i['adecuacion'].round(2)}")

# Comentarios
print()
print(" De acuerdo con lo anterior, no hay grupos subtarifados ya que la")
print(" adecuacion es mayor a 1")
print(" Esto significa que se esta cobrando más de lo que implica el riesgo ")
print(" El grupo menos sobretarifado es el de las edades 18-30")

PARAMETROS TECNICOS POR GRUPO DE EDAD
   grupo  n_polizas  n_siniestros  frecuencia     severidad    prima_pura  \
0  18-30       4802        1731.0    0.360475  35674.106340  12859.616425   
1  31-45       7559        2766.0    0.365921  32329.123506  11829.918722   
2  46-60       7579        2660.0    0.350970  34592.418094  12140.893539   
3    61+       2591         989.0    0.381706  37715.373425  14396.180748   

   prima_real_n  prima_real_t  adecuacion  adecuacion_t  loss_ratio  
0  24038.220741  29004.218709    1.869280      2.255450    0.159572  
1  24598.364731  29621.469444    2.079335      2.503945    0.133412  
2  28866.878084  34861.894458    2.377657      2.871444    0.122937  
3  33016.829024  39991.153389    2.293444      2.777900    0.119024  

        ADECUACION        
Ningun grupo esta subtarifado
Grupos sobretarifados
Grupo: 18-30  | Adecuacion 1.87
Grupo: 31-45  | Adecuacion 2.08
Grupo: 46-60  | Adecuacion 2.38
Grupo: 61+    | Adecuacion 2.29

 De acuerdo con l

---

**Fase 2 — Distribución de siniestros (SciPy)**

Ajusta lognormal a los siniestros GMM. Calcula la prima pura teórica
usando la media de la lognormal: `frec_global × media_lognormal`.
Compara con la prima pura del Fase 1.
¿Son consistentes?

---

In [6]:
# Preparamos los datos de montos
gm_montos = gm.loc[gm['monto_pagado'] > 0, 'monto_pagado'].dropna().values

# Ajustamos nuestros datos y guardamos los parametros obtenidos
sigma_ln, loc_ln, escala_ln = stats.lognorm.fit(gm_montos, floc=0)
mu_ln = np.log(escala_ln)

# Calculamos la media, mediana y el percentil al 95%
media_teo  = stats.lognorm.mean(s = sigma_ln, loc = loc_ln, scale = escala_ln)
median_teo = stats.lognorm.median(s = sigma_ln, loc = loc_ln, scale = escala_ln)
p95_teo    = stats.lognorm.ppf(0.95, s = sigma_ln, loc = loc_ln, scale = escala_ln)

print("======================================================")
print('=== PARAMETROS LOGNORMAL AJUSTADA (GASTOS MEDICOS) ===')
print("======================================================")
print(f'{"":25} {"Observado":>12} {"Teorico":>12}')
print(f'{"Media":25} ${gm_montos.mean():>11,.0f} ${media_teo:>11,.0f}')
print(f'{"Mediana":25} ${np.median(gm_montos):>11,.0f} ${median_teo:>11,.0f}')
print(f'{"Percentil 95":25} ${np.percentile(gm_montos,95):>11,.0f} ${p95_teo:>11,.0f}')
print(f'{"Percentil 95":25} ${np.percentile(gm_montos,95):>11,.0f} ${p95_teo:>11,.0f}')

# Calculamos prima pura teorica
frec_gm = float(siniestros.sum()) / len(gm)
prima_pura_ln  = frec_gm * media_teo
prima_real_neta = float(gm['prima_neta'].mean())
prima_real_total = float(gm['prima_total'].mean())

print("======================================================")
print(f'Frecuencia global GM:            {frec_gm:.4f}')
print(f'Severidad (media lognorm):      ${media_teo:,.2f}')
print(f'Prima pura teorica:             ${prima_pura_ln:,.2f} por poliza')
print(f'Prima real promedio (neta):     ${prima_real_neta:,.2f} por poliza')
print(f'Prima real promedio (total):    ${prima_real_total:,.2f} por poliza')
print(f'Adecuacion global (prima neta):  {prima_real_neta/prima_pura_ln:.3f}') # usando prima_neta
print(f'Adecuacion global (prima total): {prima_real_total/prima_pura_ln:.3f}') # Usando prima_total
print()
print(" Comparando con la prima pura calculada en la Fase 1, la prima pura teorica sí es ")
print(" consistente, pues los valores son similares. La prima pura de la fase uno ronda ")
print(" entre los 11.8k y 14.3k y la de la Fase 2 es 14.4k, por lo que son valores similares")
print(" Por otro lado, sucede algo similar con la adecuacion, pues dependiendo si se usa prima")
print(" neta o total, se encuentra entre 1.8 hasta 2.8, y en la Fase 2 ronda en valores ")
print(" similares, entre 1.8 y 2.2")

=== PARAMETROS LOGNORMAL AJUSTADA (GASTOS MEDICOS) ===
                             Observado      Teorico
Media                     $     34,452 $     39,998
Mediana                   $     19,381 $     17,147
Percentil 95              $    114,477 $    145,862
Percentil 95              $    114,477 $    145,862
Frecuencia global GM:            0.3615
Severidad (media lognorm):      $39,997.98
Prima pura teorica:             $14,461.12 por poliza
Prima real promedio (neta):     $26,882.93 por poliza
Prima real promedio (total):    $32,445.18 por poliza
Adecuacion global (prima neta):  1.859
Adecuacion global (prima total): 2.244

 Comparando con la prima pura calculada en la Fase 1, la prima pura teorica sí es 
 consistente, pues los valores son similares. La prima pura de la fase uno ronda 
 entre los 11.8k y 14.3k y la de la Fase 2 es 14.4k, por lo que son valores similares
 Por otro lado, sucede algo similar con la adecuacion, pues dependiendo si se usa prima
 neta o total, se encu

---
**Fase 3 — Simulación Monte Carlo (NumPy)**

Simula 5,000 escenarios del próximo año para GMM.
Calcula reserva P95. Compara con la prima cobrada total.
¿La cartera es suficiente?


In [7]:
# Semilla
np.random.seed(42) # reproducibilidad — siempre fijarlo antes de simular

# Usamos la distribucion binomial
# Binomial(N_polizas, probabilidad_siniestro)
binomial = np.random.binomial(n = len(gm), p = frec_gm, size = 5_000)

# Para cada siniestro, simulamos el monto usando la lognormal ajustada
# (usamos list comprehension — para cada escenario simulamos n montos y los sumamos)
pagos_sim = np.array([
    np.random.lognormal(mu_ln, sigma_ln, n).sum() if n > 0 else 0.0
    for n in binomial
])

# Calculamos estadisticas de los 5,000 escenarios
prima_cobrada_total = float(gm['prima_total'].sum())

print('=== RESULTADOS DE LA SIMULACION ===')
for ptile in [50, 75, 90, 95, 99]:
    reserva = float(np.percentile(pagos_sim, ptile))
    sufic   = prima_cobrada_total / reserva
    alerta  = '' if sufic >= 1 else '  ← INSUFICIENTE'
    print(f'  Reserva P{ptile:>2}: ${reserva:>15,.0f}  |  Suficiencia: {sufic:.1%}{alerta}')

print()
print(f'Prima cobrada total: ${prima_cobrada_total:,.0f}')
reserva_p95 = float(np.percentile(pagos_sim, 95))
print(f'Reserva P95:         ${reserva_p95:,.0f}')
print(f'Suficiencia P95:     {prima_cobrada_total/reserva_p95:.1%}')
print()

# Comentarios
print(" De acuerdo con la simulacion y la reserva calculada para el 5% de los casos adversos,")
print(" la cartera es suficiente en un 215.5%.")

=== RESULTADOS DE LA SIMULACION ===
  Reserva P50: $    325,657,306  |  Suficiencia: 224.5%
  Reserva P75: $    331,073,726  |  Suficiencia: 220.8%
  Reserva P90: $    336,192,026  |  Suficiencia: 217.4%
  Reserva P95: $    339,271,855  |  Suficiencia: 215.5%
  Reserva P99: $    345,021,388  |  Suficiencia: 211.9%

Prima cobrada total: $731,022,322
Reserva P95:         $339,271,855
Suficiencia P95:     215.5%

 De acuerdo con la simulacion y la reserva calculada para el 5% de los casos adversos,
 la cartera es suficiente en un 215.5%.


**Fase 4 — Dashboard (Plotly)**

Dashboard de 4 paneles:
- Prima pura vs prima real por grupo de edad (barras agrupadas)
- Histograma de siniestros con curva lognormal superpuesta
- Distribución de pagos Monte Carlo con líneas de percentil
- Loss ratio por estado para GMM (barras horizontales)

In [8]:
# Dividimos en cuatro subgraficos
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[
                        'Prima pura vs prima real por grupo de edad',
                        'Siniestros con curva lognormal',
                        'Distribución de pagos Monte Carlo',
                        'Loss Ratio por estado']
)

# ============ PRIMER GRAFICO ==============
# Prima pura vs prima real por grupo de edad
# ==========================================

tabla_melt = tabla_F1.melt(
    id_vars='grupo',
    value_vars=['prima_pura', 'prima_real_t'], # usando prima_total
    var_name='tipo', value_name='monto'
)

fig_barras = px.bar(
    tabla_melt,
    x='grupo', y='monto', color='tipo',
    barmode='group',
    color_discrete_map={'prima_pura':"#578017", 'prima_real_t':"#0F1974"},
    labels={'monto':'Prima (MXN)', 'grupo':'grupo de edad'},
    text_auto=',.0f'
)

for trace in fig_barras.data:
    fig.add_trace(trace, row=1, col=1)

# Ordenamos las edades
orden_edades = ['18-30', '31-45', '46-60', '61+']
fig.update_xaxes(
    categoryorder='array', 
    categoryarray=orden_edades, 
    row=1, col=1
)

# ======= SEGUNDO GRAFICO =======
# Siniestros con curva lognormal
# ===============================

# Rango del eje X
x_range = np.linspace(
    gm_montos.min(),
    np.percentile(gm_montos, 99),
    300
)
pdf_ln = stats.lognorm.pdf(x_range, sigma_ln, loc_ln, escala_ln)

# Ya que el rango del Eje X solo llega hasta el Percentil 99
# Filtramos los datos para que solo nos den hasta el percentil 99
limite_p99 = np.percentile(gm_montos, 99)
montos_filtrados = gm_montos[gm_montos <= limite_p99]

fig.add_trace(go.Histogram(
    x=montos_filtrados, nbinsx=60,
    histnorm='probability density',
    marker_color="#240CF5", opacity=0.7, name='Montos GMM',
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=x_range, y=pdf_ln, mode='lines',
    line=dict(color="#096109", width=3),
    name='Lognormal ajustada',
), row=1, col=2)

# ======== TERCER GRAFICO ==========
# Distribución de pagos Monte Carlo
# ==================================

fig.add_trace(go.Histogram(
    x = pagos_sim / 1e6,
    nbinsx = 80,
    marker_color = "#E949E9",
    opacity = 0.7,
    name = 'Simulación Monte Carlo',
), row=2, col=1) # Asignado al cuadrante inferior izquierdo

# Agregamos lineas de percentiles al grafico
for ptile, col, pos in [
    (90,  'orange',  'top left'),
    (95,  'red',     'bottom right'),
    (99,  'green', 'top right'),
]:
    val = float(np.percentile(pagos_sim, ptile)) / 1e6
    
    fig.add_vline(
        x=val, 
        line_dash='dash', 
        line_color=col,
        annotation_text=f'P{ptile}={val:.1f}M',
        annotation_position=pos,
        annotation_font_size=10,
        row=2, col=1 
    )

# Definimos el límite visual en el Percentil 99.5 de las simulaciones
#limite_eje_x = np.percentile(pagos_sim, 99.9) / 1e6
# Le aplicamos el zoom considerando limites en el eje X
#fig.update_xaxes(range=[min(pagos_sim)/1e6, limite_eje_x], row=2, col=1)

# ======== CUARTO GRAFICO ==============================
# Loss ratio por estado para GMM (barras horizontales) 
# ======================================================

# Creamos el dataframe con el loss ratio por estado
lr_estado = gm.groupby('estado')[['monto_pagado', 'prima_total']].sum().reset_index()
lr_estado['loss_ratio'] = lr_estado['monto_pagado'] / lr_estado['prima_total']

# Creamos el gráfico
colores_lr = ['#1E8449' if lr < 1 else '#C0392B' for lr in lr_estado['loss_ratio']]
fig.add_trace(go.Bar(
    x=lr_estado['loss_ratio'], y=lr_estado['estado'],
    orientation='h', marker_color=colores_lr, showlegend=False,
    hovertemplate='%{y}: LR=%{x:.4f}<extra></extra>',
), row=2, col=2)

fig.update_layout(
    height=700,
    title_text='Dashboard Actuarial — GMM Q1 2026',
    title_font_size=16,
    barmode='group',
)
# Ajustamos las etiquetas del eje Y para que no se crucen
# con el tercer grafico
fig.update_yaxes(tickfont=dict(size=9), row=2, col=2)